Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
# MAGIC        
# MAGIC
# MAGIC import trino
# MAGIC import pandas as pd
# MAGIC
# MAGIC
# MAGIC query = """
# MAGIC
# MAGIC WITH t AS (
# MAGIC   SELECT
# MAGIC     CAST(json_extract(dt.metadata, '$.correlation_id') AS VARCHAR) AS chat_reference_id,
# MAGIC     result,
# MAGIC     us.name AS "annotatedBy",
# MAGIC     rs.annotated_at AS "annotatedAt",
# MAGIC     dt.ticket_id AS ticket_id
# MAGIC   FROM (
# MAGIC     SELECT id, data_id, annotated_by, annotated_at, result, status
# MAGIC     FROM annotation.annotation_results
# MAGIC     WHERE date(CAST(split_part(REPLACE(annotated_at, 'T', ' '), '.', 1) AS TIMESTAMP) + INTERVAL '330' MINUTE) >= DATE '2025-04-08'
# MAGIC   ) rs
# MAGIC   LEFT JOIN (
# MAGIC     SELECT id, uuid, data, mime_type, batch_id, ticket_id, template_id, metadata, created_at, created_by, version, kafka_state, assignment_preferences, modified_at, modified_by, is_active, ticket_uuid
# MAGIC     FROM annotation.annotation_data
# MAGIC   ) dt ON rs.data_id = dt.id
# MAGIC   LEFT JOIN mjolnir_service.users us ON us.account_id = rs.annotated_by
# MAGIC   WHERE dt.template_id = 187
# MAGIC ),
# MAGIC parsed AS (
# MAGIC   SELECT
# MAGIC     chat_reference_id,
# MAGIC     "annotatedAt",
# MAGIC     "annotatedBy",
# MAGIC     result,
# MAGIC     CAST(json_parse(result) AS ARRAY(JSON)) AS x
# MAGIC   FROM t
# MAGIC ),
# MAGIC data AS (
# MAGIC   SELECT
# MAGIC     chat_reference_id,
# MAGIC     "annotatedBy",
# MAGIC     date("date_add"('minute', 330, "date_parse"("substring"("replace"("annotatedAt", 'T', ' '), 1, 19), '%Y-%m-%d %H:%i:%s'))) AS annotated_at_date,
# MAGIC     "date_add"('minute', 330, "date_parse"("substring"("replace"("annotatedAt", 'T', ' '), 1, 19), '%Y-%m-%d %H:%i:%s')) AS annotated_at_ist,
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') = 'category' THEN json_format(json_extract(t.x, '$.value.choices')) ELSE NULL END) AS "Category",
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') LIKE '%subcategory%' THEN json_format(json_extract(t.x, '$.value.choices')) ELSE NULL END) AS "Sub-Category",
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') = 'correctint' THEN json_format(json_extract(t.x, '$.value.choices')) ELSE NULL END) AS "Correct information as per SOP / guidelines",
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') = 'comprehenddocument' THEN json_format(json_extract(t.x, '$.value.choices')) ELSE NULL END) AS "Was it able to comprehend the uploaded document?",
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') = 'allinformation' THEN json_format(json_extract(t.x, '$.value.choices')) ELSE NULL END) AS "Given all information regarding a query (e.g. conveying 3-5 days TAT instead of saying soon)",
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') = 'alllqueries' THEN json_format(json_extract(t.x, '$.value.choices')) ELSE NULL END) AS "Addressed all queries of the customer",
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') = 'Correctinterpretation' THEN json_format(json_extract(t.x, '$.value.choices')) ELSE NULL END) AS "Correct interpretation of customer's query / message",
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') = 'latching' THEN json_format(json_extract(t.x, '$.value.choices')) ELSE NULL END) AS "Not latching to voice of customer (repeating VOC)",
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') = 'repetition' THEN json_format(json_extract(t.x, '$.value.choices')) ELSE NULL END) AS "Unique responses by the agent (No word-to-word repititions)",
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') = 'hallucinations' THEN json_format(json_extract(t.x, '$.value.choices')) ELSE NULL END) AS "No hallucinations (by GenAI Bot)",
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') = 'clarification' THEN json_format(json_extract(t.x, '$.value.choices')) ELSE NULL END) AS "Asked for customer clarification for any response that it could not understand",
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') = 'namerepeat' THEN json_format(json_extract(t.x, '$.value.choices')) ELSE NULL END) AS "Saying customer's name only once (not repeating it)",
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') = 'Rushing' THEN json_format(json_extract(t.x, '$.value.choices')) ELSE NULL END) AS "Not Rushing on call",
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') = 'Interruption' THEN json_format(json_extract(t.x, '$.value.choices')) ELSE NULL END) AS "Not Interrupting on call",
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') = 'jargons' THEN json_format(json_extract(t.x, '$.value.choices')) ELSE NULL END) AS "Replies are easy to understand (no technical terms / jargons)",
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') = 'dragging' THEN json_format(json_extract(t.x, '$.value.choices')) ELSE NULL END) AS "Not Dragging the call unneccesarily / active listening",
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') = 'unnecessary' THEN json_format(json_extract(t.x, '$.value.choices')) ELSE NULL END) AS "No Unnecessary information (answer not relevant to questions asked / entertaining random queries not related to Navi)",
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') = 'loop' THEN json_format(json_extract(t.x, '$.value.choices')) ELSE NULL END) AS "Did not go on a loop when customer said Thanks / Ok / any other acknowledgement",
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') = 'lang' THEN json_format(json_extract(t.x, '$.value.choices')) ELSE NULL END) AS "Adherence to customer language preference",
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') = 'understanding' THEN json_format(json_extract(t.x, '$.value.choices')) ELSE NULL END) AS "Demonstrated understanding of customer emotions and concerns",
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') = 'furtherassistance' THEN json_format(json_extract(t.x, '$.value.choices')) ELSE NULL END) AS "Seeked further assistance",
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') = 'closure' THEN json_format(json_extract(t.x, '$.value.choices')) ELSE NULL END) AS "Closing script adherence (Correct CSAT script)",
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') = 'opening and closing' THEN json_format(json_extract(t.x, '$.value.choices')) ELSE NULL END) AS "Adhered standard Opening / Closing",
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') = 'Professional' THEN json_format(json_extract(t.x, '$.value.choices')) ELSE NULL END) AS "Professional behaviour with customer(not referring to customers with slangs like bhai, yaar etc.)",
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') = 'escalated agent' THEN json_format(json_extract(t.x, '$.value.choices')) ELSE NULL END) AS "Correctly escalated to an agent",
# MAGIC     max(CASE WHEN json_extract_scalar(t.x, '$.from_name') = 'remarks' THEN json_format(json_extract(t.x, '$.value.text')) ELSE NULL END) AS "Remarks"
# MAGIC   FROM parsed AS p, unnest(x) AS t(x)
# MAGIC   WHERE date("date_add"('minute', 330, "date_parse"("substring"("replace"("annotatedAt", 'T', ' '), 1, 19), '%Y-%m-%d %H:%i:%s'))) BETWEEN (CURRENT_DATE - INTERVAL '89' DAY) AND CURRENT_DATE
# MAGIC   GROUP BY 1, 2, 3, 4
# MAGIC )
# MAGIC SELECT DISTINCT a.*, b.csat
# MAGIC FROM data a
# MAGIC LEFT JOIN (
# MAGIC   SELECT canonical_ticket_id, avg(csat) AS csat
# MAGIC   FROM corp_analytics.cx_sub_ticket_data_model_tbl
# MAGIC   GROUP BY 1
# MAGIC ) b ON a.chat_reference_id = b.canonical_ticket_id
# MAGIC ORDER BY annotated_at_date DESC
# MAGIC
# MAGIC """
# MAGIC
# MAGIC df  = get_sql_df(query)
# MAGIC if not df.empty:
# MAGIC     display(df)
# MAGIC else:
# MAGIC     print(f"No annotation data found for today ({pd.Timestamp.now().strftime('%Y-%m-%d')}). The DataFrame has {len(df.columns)} columns and 0 rows.")
# MAGIC     display(spark.createDataFrame([], schema="dummy STRING").limit(0))

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/17l7Sjlwp3tvjj-zVvm6EvqDhwb5QpyKfFRTl7siP0Vc/edit?gid=0#gid=0"
worksheet_name = "PL Voice Bot"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=1
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=1
)